> Os exemplos desta página usam dados sintéticos gerados pelo projeto. Não são observações ASTER nem permitem interpretar valores como reflectância, temperatura ou classes reais. Veja [os dados de exemplo](../../../data/README.md).



# Aplicação piloto — PCA + k-means

Demonstração com dados sintéticos. Para observações reais, ajuste `RASTER_PATH` com um arquivo local cuja origem e condições de uso estejam documentadas.
Saídas em `projetos/pca-e-agrupamento/figuras/`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_utils").is_dir())
PROJETO = REPO / "projetos/pca-e-agrupamento"

RASTER_PATH = REPO / 'data/exemplos/exemplo_sintetico.tif'
# RASTER_PATH = PROJETO / 'data' / 'seu_raster.tif'
OUT = PROJETO / 'figuras'
OUT.mkdir(parents=True, exist_ok=True)
print(RASTER_PATH)

In [ ]:
with rasterio.open(RASTER_PATH) as ds:
    img = ds.read().astype(np.float64)
bands, rows, cols = img.shape
X = img.reshape(bands, -1).T
Xs = StandardScaler().fit_transform(X)

pca = PCA(n_components=min(3, bands), random_state=0)
scores = pca.fit_transform(Xs)
print('variância explicada', pca.explained_variance_ratio_)

idx = np.random.default_rng(0).choice(len(scores), min(25000, len(scores)), replace=False)
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(scores[idx])
labels = km.predict(scores).reshape(rows, cols)

pc_img = scores.T.reshape(3, rows, cols)
rgb = np.stack([(pc_img[i] - pc_img[i].min()) / (np.ptp(pc_img[i]) + 1e-9) for i in range(3)], axis=-1)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(np.clip(rgb, 0, 1)); ax[0].set_title('PCA RGB')
ax[1].imshow(labels, cmap='tab10'); ax[1].set_title('k-means (PCs)')
for a in ax: a.axis('off')
fig.tight_layout()
fig.savefig(OUT / 'piloto_pca_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()
print('salvo', OUT / 'piloto_pca_kmeans.png')